# Qwen3 embedding 微调，使用官方提供的脚本

多卡 up给的

使用环境变量设置路径:
- LLM_AGENT_MODEL_DIR: 模型目录
- LLM_AGENT_OUTPUT_DIR: 输出目录

INFONCE_MASK_FAKE_NEGATIVE=true  
CUDA_VISIBLE_DEVICES=6,7 \
NPROC_PER_NODE=2 \
swift sft \
    --model ${LLM_AGENT_MODEL_DIR:-/data/llm_agent/models}/Qwen3-Embedding-4B/Qwen/Qwen3-Embedding-4B \
    --task_type embedding \
    --model_type qwen3_emb \
    --train_type lora \
    --dataset ${LLM_AGENT_OUTPUT_DIR:-/data/llm_agent/output}/SFT_emb_reranker/data/emb_train.jsonl \
    --split_dataset_ratio 0.05 \
    --eval_strategy steps \
    --output_dir output \
    --eval_steps 20 \
    --num_train_epochs 5 \
    --save_steps 50 \
    --per_device_train_batch_size 1 \
    --per_device_eval_batch_size 1 \
    --gradient_accumulation_steps 8 \
    --learning_rate 2e-5 \
    --loss_type infonce \
    --label_names labels \
    --dataloader_drop_last true \
    --output_dir ${LLM_AGENT_OUTPUT_DIR:-/data/llm_agent/output}/SFT_emb_reranker/lora_emb \
    --use_liger_kernel false \
    --attn_impl flash_attn

官方提供的 但是用不了 deepspeed

使用环境变量设置路径:
- LLM_AGENT_MODEL_DIR: 模型目录
- LLM_AGENT_OUTPUT_DIR: 输出目录

CUDA_VISIBLE_DEVICES=6,7 \
INFONCE_TEMPERATURE=0.1 \
NPROC_PER_NODE=2 \
swift sft \
    --model ${LLM_AGENT_MODEL_DIR:-/data/llm_agent/models}/Qwen3-Embedding-4B/Qwen/Qwen3-Embedding-4B \
    --task_type embedding \
    --tuner_type lora \
    --lora_rank 8 \
    --lora_alpha 32 \
    --learning_rate 5e-5 \
    --target_modules all-linear \
    --dataset ${LLM_AGENT_OUTPUT_DIR:-/data/llm_agent/output}/SFT_emb_reranker/data/emb_train.jsonl \
    --attn_impl flash_attn \
    --padding_free true \
    --torch_dtype bfloat16 \
    --load_from_cache_file true \
    --split_dataset_ratio 0.05 \
    --eval_strategy steps \
    --output_dir ${LLM_AGENT_OUTPUT_DIR:-/data/llm_agent/output}/SFT_emb_reranker/lora_emb \
    --save_steps 50 \
    --eval_steps 50 \
    --save_total_limit 2 \
    --logging_steps 5 \
    --num_train_epochs 5 \
    --max_length 8192 \
    --per_device_train_batch_size 8 \
    --per_device_eval_batch_size 8 \
    --gradient_accumulation_steps 1 \
    --dataloader_num_workers 4 \
    --dataset_num_proc 4 \
    --warmup_ratio 0.05 \
    --loss_type infonce \
    --dataloader_drop_last true \
    --report_to swanlab \
    --swanlab_project qwen3_4b_embedding_lora \
    --swanlab_exp_name lora_emb_run1 \
    --swanlab_workspace zeeoreone \
    --swanlab_mode cloud

合并模型

使用环境变量设置路径:
- LLM_AGENT_OUTPUT_DIR: 输出目录 (包含 LoRA 适配器)
- LLM_AGENT_MODEL_DIR: 模型目录 (合并后的模型保存位置)

swift export \
    --adapters /root/autodl-tmp/ms-swift/output/v2-20250619-151039/checkpoint-155 \
    --merge_lora true


swift export \
    --adapters ${LLM_AGENT_OUTPUT_DIR:-/data/llm_agent/output}/SFT_emb_reranker/lora_emb/v0-20260329-210551/checkpoint-215 \
    --merge_lora true \
    --output_dir ${LLM_AGENT_MODEL_DIR:-/data/llm_agent/models}/Qwen3-Embedding-4B-IC \
    --torch_dtype bfloat16 \
    --safe_serialization true \
    --max_shard_size 5GB

# Qwen3 Reranker 微调，使用官方提供的脚本

https://github.com/QwenLM/Qwen3-Embedding/issues/75

官方提供的 但是用不了 deepspeed

使用环境变量设置路径:
- LLM_AGENT_MODEL_DIR: 模型目录
- LLM_AGENT_OUTPUT_DIR: 输出目录

CUDA_VISIBLE_DEVICES=0,1 \
NPROC_PER_NODE=2 \
swift sft \
    --model ${LLM_AGENT_MODEL_DIR:-/data/llm_agent/models}/Qwen3-Reranker-4B \
    --task_type generative_reranker \
    --loss_type pointwise_reranker \
    --tuner_type lora \
    --lora_rank 8 \
    --lora_alpha 32 \
    --learning_rate 5e-5 \
    --target_modules all-linear \
    --dataset ${LLM_AGENT_OUTPUT_DIR:-/data/llm_agent/output}/SFT_emb_reranker/data/reranker_train_offical.jsonl \
    --attn_impl flash_attn \
    --padding_free true \
    --torch_dtype bfloat16 \
    --load_from_cache_file true \
    --split_dataset_ratio 0.02 \
    --eval_strategy steps \
    --output_dir ${LLM_AGENT_OUTPUT_DIR:-/data/llm_agent/output}/SFT_emb_reranker/lora_reranker \
    --save_steps 50 \
    --eval_steps 50 \
    --save_total_limit 2 \
    --logging_steps 5 \
    --num_train_epochs 1 \
    --max_length 4096 \
    --per_device_train_batch_size 1 \
    --per_device_eval_batch_size 1 \
    --gradient_accumulation_steps 8 \
    --dataloader_num_workers 4 \
    --dataset_num_proc 4 \
    --warmup_ratio 0.05 \
    --dataloader_drop_last true \
    --report_to swanlab \
    --swanlab_project qwen3_4b_reranker_lora \
    --swanlab_exp_name lora_reranker_run1 \
    --swanlab_workspace zeeoreone \
    --swanlab_mode cloud

使用环境变量设置路径:
- LLM_AGENT_OUTPUT_DIR: 输出目录 (包含 LoRA 适配器)
- LLM_AGENT_MODEL_DIR: 模型目录 (合并后的模型保存位置)

swift export \
    --adapters ${LLM_AGENT_OUTPUT_DIR:-/data/llm_agent/output}/SFT_emb_reranker/lora_reranker/v0-20260329-220200/checkpoint-45 \
    --merge_lora true \
    --output_dir ${LLM_AGENT_MODEL_DIR:-/data/llm_agent/models}/Qwen3-Reranker-4B-IC \
    --torch_dtype bfloat16 \
    --safe_serialization true \
    --max_shard_size 5GB